# oxDNA: LAMMPS Propeller Twist Optimization

This notebook optimizes oxDNA1 force-field parameters to match a target propeller twist using LAMMPS as the simulation engine and DiffTRe for gradient computation.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

This notebook shows how to use LAMMPS as an alternative simulation backend for oxDNA model optimization. LAMMPS provides a fast, well-tested MD engine while mythos handles the differentiable optimization on top.

## Imports

In [ ]:
import logging
from pathlib import Path

import jax
import jax.numpy as jnp
import mythos.energy.dna1 as dna1_energy
import mythos.observables as jd_obs
import mythos.optimization.objective as jdna_objective
import mythos.optimization.optimization as jdna_optimization
import optax
from mythos.input import topology
from mythos.simulators.lammps.lammps_oxdna import LAMMPSoxDNASimulator
from mythos.ui.loggers.console import ConsoleLogger

jax.config.update("jax_enable_x64", True)
logging.basicConfig(level=logging.INFO)

## Configuration

In [ ]:
N_OPT_STEPS = 25
LEARNING_RATE = 1e-3
INPUT_DIR = Path("../../data/templates/simple-helix-60bp-oxdna1-lammps").resolve()
TARGET = 25 #jd_obs.propeller.TARGETS["oxDNA"]
KT = 0.1  # Must match the value used in LAMMPS

## Build the energy function and simulator

Note that the LAMMPS Simulator for oxDNA has requirements for its build and
installation (e.g. `CG-DNA` package), refer to the [LAMMPS
documentation](https://docs.lammps.org/) for details.

Note also that due to the potential complexity afforded by the script-like
nature of the LAMMPS input file, variables overrides are currently supported
only if a corresponding variable is defined in the input script. In this
example, the input script defines variables that make sense for tuning or
required by DiffTRe (kt), ex:

```
variable ofreq equal 10000
variable steps equal 250000
variable kt equal 0.1
```

And we override them using the `variables` paramerter.

In [ ]:
from jax_md import space


top = topology.from_oxdna_file(INPUT_DIR / "sys.top")

energy_fn = dna1_energy.create_default_energy_fn(
    topology=top,
    displacement_fn=space.periodic(200.0)[0],  # box size defined in lammps init configuration
).with_noopt(
    "ss_stack_weights", "ss_hb_weights"
).without_terms(
    "BondedExcludedVolume"
)

simulator = LAMMPSoxDNASimulator(
    name="lammps-oxdna",
    input_dir=INPUT_DIR,
    energy_fn=energy_fn,
    variables={
        "steps": 250_000,
        "ofreq": 1_000,
        "kt": KT,
    }
)

## Define the propeller twist objective

DiffTre requires the simulator trajectory to contain a valid temperature value,
which will be written by LAMMPS in the case it has a variable defined. See note
above regarding variable definitions in the LAMMPS input script.

In [ ]:
def get_h_bonded_base_pairs(n_nucs_per_strand: int) -> jnp.ndarray:
    s1_nucs = list(range(n_nucs_per_strand))
    s2_nucs = list(range(n_nucs_per_strand, n_nucs_per_strand * 2))
    s2_nucs.reverse()
    return jnp.array(list(zip(s1_nucs, s2_nucs, strict=True)), dtype=jnp.int32)

h_bonded_pairs = get_h_bonded_base_pairs(top.n_nucleotides // 2)

prop_twist_fn = jd_obs.propeller.PropellerTwist(
    rigid_body_transform_fn=energy_fn.energy_fns[0].transform_fn,
    h_bonded_base_pairs=h_bonded_pairs,
)

def prop_twist_loss_fn(traj, weights, *_args, **_kwargs):
    obs = prop_twist_fn(traj)
    expected_prop_twist = jnp.dot(weights, obs)
    loss = jnp.sqrt((expected_prop_twist - TARGET) ** 2)
    return loss, (("prop_twist", expected_prop_twist), {})


opt_params = energy_fn.opt_params()

propeller_twist_objective = jdna_objective.DiffTReObjective(
    name="DiffTRe",
    required_observables=simulator.exposes(),
    logging_observables=["loss", "prop_twist"],
    grad_or_loss_fn=prop_twist_loss_fn,
    energy_fn=energy_fn,
    min_n_eff_factor=0.95,
    n_equilibration_steps=100,  # Note that this is number of output frames to remove, not simulation steps!
    max_valid_opt_steps=100,
)

## Setup logging

Mythos defines a simple logging API that is automatically available when using
the `Optimizer.run()` method, and available for manual logging via its methods
such as `log_metric()`. 

Here, we demonstrate the `JupyterLogger` to log metrics via plotly to a
dashboard embedded in the notebook (if we are in a jupyter environment). Note
that the optimizer will log metrics for each observable, and thus qualifies the
metric name with the observable name to prevent collisions
(`<observable_name>.<metric_name>`) e.g. `DiffTre.loss` and
`DiffTre.prop_twist`.

In [ ]:
has_jupyter = False
try:
    get_ipython()
    has_jupyter = True
except NameError:
    pass

if has_jupyter:
    from mythos.ui.loggers.jupyter import JupyterLogger

    logger = JupyterLogger(
        simulators=[simulator.name],
        objectives=[propeller_twist_objective.name],
        observables=simulator.exposes(),
        metrics_to_log=["DiffTRe.loss", "DiffTRe.prop_twist"],
        max_opt_steps=N_OPT_STEPS,
    )
else:
    logger = ConsoleLogger()

## Run the optimization

Using `SimpleOptimizer.run()` to manage the optimization loop.

In [ ]:


opt = jdna_optimization.SimpleOptimizer(
    objective=propeller_twist_objective,
    simulator=simulator,
    optimizer=optax.adam(learning_rate=LEARNING_RATE),
    logger=logger,
)

output = opt.run(params=opt_params, n_steps=N_OPT_STEPS)
print("\nOptimization complete!")
print(f"Final params: {output.opt_params}")